# 05 分層分析與交絡因子 — 參考解答

松柏護理之家退伍軍人症群聚事件分層分析練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## 題目 1：水療使用的交絡分析

In [ ]:
# --- 粗 RR ---
ct_hydro = pd.crosstab(df["hydrotherapy_use"], df["infected"])
a0 = int(ct_hydro.loc[1, 1])
b0 = int(ct_hydro.loc[1, 0])
c0 = int(ct_hydro.loc[0, 1])
d0 = int(ct_hydro.loc[0, 0])
crude_rr_hydro = risk_ratio(a0, a0 + b0, c0, c0 + d0)
print(f"粗 RR (hydrotherapy → infected) = {crude_rr_hydro:.3f}")

# --- 驗證交絡條件 ---
print("\n=== 功能狀態 × 水療使用率 ===")
print(pd.crosstab(df["functional_status"], df["hydrotherapy_use"],
                  normalize="index").round(3))

# --- 分層 RR ---
strata = sorted(df["functional_status"].unique())
hydro_results = []

for s in strata:
    sub = df[df["functional_status"] == s]
    ct_s = pd.crosstab(sub["hydrotherapy_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {s}: 跳過")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    hydro_results.append({
        "stratum": s, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

hydro_df = pd.DataFrame(hydro_results)
print("\n=== 分層 RR ===")
for _, row in hydro_df.iterrows():
    print(f"  {row['stratum']:20s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})")
print(f"\n  粗 RR = {crude_rr_hydro:.3f}")

## 題目 2：Mantel-Haenszel 調整

In [ ]:
numerator = 0
denominator = 0

for _, row in hydro_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh_hydro = numerator / denominator

print(f"Mantel-Haenszel 調整後 RR = {rr_mh_hydro:.3f}")
print(f"粗 RR                     = {crude_rr_hydro:.3f}")
print(f"差異                      = {crude_rr_hydro - rr_mh_hydro:.3f}")

if abs(crude_rr_hydro - rr_mh_hydro) > 0.1:
    print("\n→ 功能狀態確實是水療使用的交絡因子（粗 RR 被膨脹）")
else:
    print("\n→ 控制功能狀態後 RR 變化不大，交絡效應有限")

## 題目 3（挑戰題）：按年齡組分層 + 森林圖

In [ ]:
# 粗 RR
ct_shower = pd.crosstab(df["shower_use"], df["infected"])
a_crude = int(ct_shower.loc[1, 1])
b_crude = int(ct_shower.loc[1, 0])
c_crude = int(ct_shower.loc[0, 1])
d_crude = int(ct_shower.loc[0, 0])
crude_rr = risk_ratio(a_crude, a_crude + b_crude, c_crude, c_crude + d_crude)

# 建立年齡組
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# 分層 RR
age_results = []
for grp in ["60-69", "70-79", "80-89", "90+"]:
    sub = df[df["age_group"] == grp]
    ct_s = pd.crosstab(sub["shower_use"], sub["infected"])
    if ct_s.shape != (2, 2):
        print(f"  {grp}: 跳過")
        continue
    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s
    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)
    age_results.append({
        "stratum": grp, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

age_df = pd.DataFrame(age_results)
print("=== 按年齡組分層 RR ===")
for _, row in age_df.iterrows():
    print(f"  {row['stratum']:10s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")

In [ ]:
# 森林圖
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(age_df))

ax.errorbar(
    age_df["RR"], y_pos,
    xerr=[age_df["RR"] - age_df["CI_lower"],
          age_df["CI_upper"] - age_df["RR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_rr, color="red", linestyle=":", alpha=0.7,
           label=f"粗 RR={crude_rr:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(age_df["stratum"])
ax.set_xlabel("Risk Ratio (RR)")
ax.set_title("森林圖：淋浴使用 → 感染（按年齡組分層）")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# MH 調整後 RR
num = 0
den = 0
for _, row in age_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    num += a_i * (c_i + d_i) / n_i
    den += c_i * (a_i + b_i) / n_i

rr_mh_age = num / den

print(f"MH 調整後 RR（控制年齡） = {rr_mh_age:.3f}")
print(f"粗 RR                    = {crude_rr:.3f}")
print(f"差異                     = {crude_rr - rr_mh_age:.3f}")

# 同質性
rr_vals = age_df["RR"].values
print(f"\n各層 RR 範圍：{rr_vals.min():.3f} – {rr_vals.max():.3f}")
if rr_vals.max() - rr_vals.min() > 0.5:
    print("→ 各年齡組 RR 差異較大，可能存在年齡的效果修飾")
else:
    print("→ 各年齡組 RR 相近，年齡的交互作用不明顯")

### 解讀

- **功能狀態**：臥床住民不淋浴也較少感染，能行走的住民淋浴率高也較多感染 → 經典交絡
- **MH 調整後**：如果 RR_MH 明顯小於粗 RR，確認功能狀態是交絡因子
- **年齡分層**：如果各年齡組的 RR 接近，年齡交互作用不大
- **限制**：分層分析一次只能控制一個變項 → 需要 Ch06 邏輯斯迴歸同時調整多個因子